# 01 — EDA y diagnóstico

Este notebook documenta la estructura de los datos, calidad, patrón semanal, diferencias por producto y relación entre tamaño de tienda y demanda.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd()
if not (ROOT / "data").exists(): ROOT = ROOT.parent
DATA = ROOT / "data" / "raw"
OUTPUT = ROOT / "outputs"


In [ ]:
ventas=pd.read_csv(DATA/'ventas_historicas.csv',parse_dates=['fecha'])
inventario=pd.read_csv(DATA/'inventario_actual.csv')
catalogo=pd.read_csv(DATA/'catalogo_productos.csv')
tienda=pd.read_csv(DATA/'maestro_tiendas.csv')
print('Ventas:', ventas.shape)
print('Periodo:', ventas.fecha.min().date(), 'a', ventas.fecha.max().date())
print('Series SKU-tienda:', ventas[['id_tienda','id_producto']].drop_duplicates().shape[0])
print('Nulos:', ventas.isna().sum().sum(), 'Duplicados:', ventas.duplicated().sum())


In [ ]:
ventas['unidades_vendidas'].describe()

In [ ]:
dow=ventas.assign(dow=ventas.fecha.dt.dayofweek).groupby('dow').unidades_vendidas.mean()
print(dow.rename({0:'Lun',1:'Mar',2:'Mié',3:'Jue',4:'Vie',5:'Sáb',6:'Dom'}))
print('Ratio fin de semana vs lunes-jueves:', dow.loc[4:6].mean()/dow.loc[0:3].mean())

In [ ]:
prod=ventas.groupby('id_producto').unidades_vendidas.agg(['mean','std','sum']).sort_values('mean',ascending=False)
prod

In [ ]:
tmp=ventas.groupby('id_tienda').unidades_vendidas.mean().rename('demanda_media').reset_index().merge(tienda,on='id_tienda')
print('Correlación tamaño-demanda:', tmp.demanda_media.corr(tmp['tamaño_m2']))
tmp.sort_values('demanda_media',ascending=False)

In [ ]:
# Cobertura aproximada con el inventario actual: stock / (demanda diaria * 7)
avg=ventas.groupby(['id_tienda','id_producto']).unidades_vendidas.mean().reset_index(name='demanda_dia')
cov=avg.merge(inventario,on=['id_tienda','id_producto'])
cov['semanas_cobertura']=cov.stock_actual/(cov.demanda_dia*7)
print(cov.semanas_cobertura.describe())


### Conclusiones
- La estacionalidad semanal es más informativa que una tendencia anual, dado que solo hay 91 días.
- Los lags 7, 14 y 21 son candidatos naturales para el forecast.
- El tamaño de tienda puede aportar señal, pero se interpreta como predictor y no como causalidad.
- La cobertura de inventario muestra por qué el forecast debe terminar en una decisión de pedido.